In [1]:
!pip install spacy scikit-learn pandas PyPDF2 numpy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 51.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
import PyPDF2
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files
import io

In [3]:
nlp = spacy.load("en_core_web_sm")

In [ ]:


uploaded = files.upload()

In [53]:
def extract_text_from_pdf(file):
    text = ""
    pdf_reader = PyPDF2.PdfReader(io.BytesIO(file))
    for page in pdf_reader.pages:
        if page.extract_text():
            text += page.extract_text()
    return text

In [54]:
def preprocess_text(text):
    doc = nlp(text.lower())
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct
    ]
    return " ".join(tokens)

In [55]:
job_description = """
Looking for a Data Analyst skilled in Python, SQL, Machine Learning,
Power BI, Data Visualization, Statistics, and Excel.
Experience with Pandas, NumPy and Scikit-learn preferred.
"""

In [56]:
resume_texts = []
filenames = []

for filename in uploaded.keys():
    file_content = uploaded[filename]
    text = extract_text_from_pdf(file_content)
    processed_text = preprocess_text(text)

    resume_texts.append(processed_text)
    filenames.append(filename)

In [57]:
documents = [preprocess_text(job_description)] + resume_texts

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()

In [58]:
important_keywords = {
    "python": 5,
    "sql": 5,
    "machine learning": 4,
    "power bi": 3,
    "statistics": 3,
    "excel": 2
}

def keyword_boost_score(resume_text):
    score = 0
    for keyword, weight in important_keywords.items():
        if keyword in resume_text:
            score += weight
    return score

In [59]:
final_scores = []

for i, resume_text in enumerate(resume_texts):
    keyword_score = keyword_boost_score(resume_text)
    final_score = (similarity_scores[i] * 0.7) + (keyword_score * 0.3)
    final_scores.append(final_score)

In [60]:
results = list(zip(filenames, final_scores))
results = sorted(results, key=lambda x: x[1], reverse=True)

ranking_df = pd.DataFrame(results, columns=["Resume", "Score"])
ranking_df

,Resume,Score
0,DOC-20250704-WA0001. (2).pdf,3.762527


In [61]:
ranking_df.to_csv("HR_Report.csv", index=False)
files.download("HR_Report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>